###Build Races Dimension
1. Read silver races table
2. Read silver circuits table
3. Join the data from races with circuits using circuits_id
4. Select the required columns
5. Write the transformed data to gold dim_races table

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-Common/01.environment-config

In [0]:
%run ../00-Common/04.gold_helpers

In [0]:
from pyspark.sql import functions as f

In [0]:
target_table = f"{catalog_name}.{gold_schema}.dim_races"

In [0]:
circuits_df  = spark.table(f"{catalog_name}.{silver_schema}.circuits").filter(f.col("batch_id") == v_batch_id)
races_df = spark.table(f"{catalog_name}.{silver_schema}.races").filter(f.col("batch_id") == v_batch_id)

In [0]:
dim_races_df = (
    races_df
        .join(
            circuits_df,
            races_df.circuit_id == circuits_df.circuit_id,
            "inner"
            )
        .select(
            races_df.season,
            races_df.round,
            races_df.race_name,
            races_df.race_date,
            circuits_df.circuit_name,
            circuits_df.locality,
            circuits_df.country
        )
)

In [0]:
#display(dim_races_df)

In [0]:
write_to_gold(
    dim_races_df,
    target_table,
    "t.season = s.season AND t.round = s.round",
    [
        "race_name",
        "race_date",
        "circuit_name",
        "locality",
        "country"
    ]
)

In [0]:
display(spark.table(target_table))